# MongoDB Atlas: sample_mflix.movies 
**Objetivo:** practicar consultas `find()` y agregaciones `aggregate()` sobre la base de datos `sample_mflix`, colección `movies`.




## 0) Instalación y requisitos
Si no tienes las librerías, ejecuta:

```bash
pip install pymongo pandas dnspython
```

Necesitas:
- Un cluster en MongoDB Atlas.
- Tu IP permitida en **Network Access**.
- Un usuario con permisos de lectura en `sample_mflix`.


## 1) Conexión a MongoDB Atlas (segura)
Guarda tu URI en una variable de entorno `MONGODB_URI`.

Ejemplo (NO pegues contraseña real en el notebook):
```
mongodb+srv://<user>:<password>@<cluster>.mongodb.net/?retryWrites=true&w=majority
```


In [ ]:
import os
import certifi
import pandas as pd
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

uri = ""

client = MongoClient(uri, server_api=ServerApi('1'), tlsCAFile=certifi.where())

client.admin.command('ping')
print(' Conexión OK')

db    = client['sample_mflix']
colls = db.list_collection_names()
print(colls)

col = db['movies']
print('Colección:', col.full_name)


 Conexión OK
['users', 'comments', 'sessions', 'movies', 'embedded_movies', 'theaters']
Colección: sample_mflix.movies


## Helpers
Funciones pequeñas para convertir resultados a DataFrame y previsualizar documentos.


In [2]:
from pprint import pprint

def to_df(cursor_or_list):
    return pd.DataFrame(list(cursor_or_list))

def head_docs(cursor, n=3):
    for i, doc in enumerate(cursor):
        if i >= n:
            break
        pprint(doc)


## Ejercicios
Cada ejercicio incluye enunciado + pista + solución.


### 1) Conteo total de películas
**Enunciado:** ¿Cuántos documentos (películas) hay en `movies`?

**Pista:** `count_documents({})`.


In [ ]:
# Contamos todos los documentos de la colección 'movies'
total_movies = col.count_documents({})

print(f"Total películas: {total_movies}")

Total películas: 21349


### 2) Top 5 películas más recientes (campos seleccionados)
**Enunciado:** Devuelve 5 películas más recientes mostrando `title`, `year`, `genres`, `imdb.rating`.

**Pista:** `find({}, projection).sort('year', -1).limit(5)`.


In [4]:
# Definimos los campos que queremos ver (1 = mostrar, 0 = ocultar)
projection = {
    "genres": 1, 
    "title": 1, 
    "year": 1, 
    "imdb.rating": 1, 
    "_id": 0
}

# Hacemos la consulta: buscar todo, proyectar, ordenar por año desc, limitar a 5
cursor = col.find({}, projection).sort("year", -1).limit(5)

# Convertimos a DataFrame y mostramos
display(to_df(cursor))

,genres,title,year,imdb
0,[Documentary],The Roosevelts: An Intimate History,2014è,{'rating': 8.8}
1,"[Action, Adventure, Sci-Fi]",Halo: Nightfall,2014è,{'rating': 5.7}
2,[Documentary],The Weight of the Nation,2012è,{'rating': 7.4}
3,[Drama],Hit & Miss,2012è,{'rating': 7.9}
4,[Mystery],Penance,2012è,{'rating': 7.2}


### 3) Filtrar por género + año y ordenar por rating
**Enunciado:** Top 10 películas `Drama` desde 2010, ordenadas por `imdb.rating` desc y `year` desc.

**Pista:** `{'genres': 'Drama', 'year': {'$gte': 2010}}`.


In [5]:
# Consulta: Género Drama y año >= 2010
query = {
    "genres": "Drama",
    "year": {"$gte": 2010}
}

# Proyección exacta para tu salida
projection = {"genres": 1, "title": 1, "year": 1, "imdb": 1, "_id": 0}

# Ordenamos por imdb.rating descendente (-1)
# Nota: Esto pone los strings vacíos "" antes que los números
cursor = col.find(query, projection).sort([("imdb.rating", -1), ("year", -1)]).limit(10)

display(to_df(cursor))

,genres,title,year,imdb
0,"[Biography, Drama]",The Danish Girl,2015,"{'rating': '', 'votes': '', 'id': 810819}"
1,"[Comedy, Drama, Romance]",Manhattan Romance,2015,"{'rating': '', 'votes': '', 'id': 2608324}"
2,"[Drama, Horror, Mystery]",The Childhood of a Leader,2015,"{'rating': '', 'votes': '', 'id': 2815902}"
3,"[Documentary, Drama, History]",Catching the Sun,2015,"{'rating': '', 'votes': '', 'id': 1698654}"
4,"[Crime, Drama, Mystery]",A Bigger Splash,2015,"{'rating': '', 'votes': '', 'id': 2056771}"
5,"[Drama, Mystery, Thriller]",The Ghost and the Whale,2015,"{'rating': '', 'votes': '', 'id': 2429278}"
6,[Drama],Desierto,2015,"{'rating': '', 'votes': '', 'id': 3147312}"
7,[Drama],Outliving Emily,2015,"{'rating': '', 'votes': '', 'id': 3000308}"
8,"[Drama, Sci-Fi, Thriller]",Ad Inexplorata,2015,"{'rating': '', 'votes': '', 'id': 2674430}"
9,"[Drama, Sci-Fi, Thriller]",Man Down,2015,"{'rating': '', 'votes': '', 'id': 2461520}"


### 4) Años con más películas (aggregate)
**Enunciado:** Cuenta películas por `year` y devuelve top 10 años con más películas.

**Pista:** `$group` por `'$year'` y `$sum: 1`.


In [6]:
pipeline = [
    {"$group": {"_id": "$year", "n": {"$sum": 1}}}, # Agrupar por año
    {"$sort": {"n": -1}},                           # Ordenar por cantidad
    {"$limit": 10},                                 # Top 10
    {"$project": {"year": "$_id", "n": 1, "_id": 0}} # Formato de salida
]

df_years = to_df(col.aggregate(pipeline))
display(df_years)

,n,year
0,1105,2013
1,1073,2014
2,955,2012
3,917,2009
4,893,2011
5,886,2008
6,866,2010
7,810,2007
8,774,2006
9,713,2005


### 5) Distribución por clasificación (`rated`)
**Enunciado:** Cuenta cuántas películas hay por `rated`.

**Pista:** `$group` por `'$rated'`.


In [7]:
pipeline = [
    {"$group": {"_id": "$rated", "n": {"$sum": 1}}},
    {"$sort": {"n": -1}}, # Orden descendente
    {"$project": {"rated": "$_id", "n": 1, "_id": 0}}
]

df_rated = to_df(col.aggregate(pipeline))
display(df_rated)

,n,rated
0,9894,None
1,5537,R
2,2321,PG-13
3,1852,PG
4,709,APPROVED
5,477,G
6,181,PASSED
7,89,TV-14
8,76,TV-PG
9,60,TV-MA


### 6) Rating medio por década (con mínimo de películas)
**Enunciado:** Calcula media de `imdb.rating` por década y muestra solo décadas con ≥100 películas.

**Pista:** `decade = year - (year % 10)` con `$subtract` y `$mod`.


In [8]:
pipeline = [
    # 1. Filtramos para asegurar que son números (evitamos strings vacíos aquí)
    {"$match": {
        "year": {"$type": "number"}, 
        "imdb.rating": {"$type": "number"}
    }},
    
    # 2. Calculamos la década
    {"$project": {
        "decade": {
            "$subtract": ["$year", {"$mod": ["$year", 10]}]
        },
        "rating": "$imdb.rating"
    }},
    
    # 3. Agrupamos
    {"$group": {
        "_id": "$decade",
        "n": {"$sum": 1},
        "avg_rating": {"$avg": "$rating"}
    }},
    
    # 4. Filtro: al menos 100 películas
    {"$match": {"n": {"$gte": 100}}},
    
    # 5. Orden cronológico
    {"$sort": {"_id": 1}},
    
    # 6. Proyección final
    {"$project": {"decade": "$_id", "n": 1, "avg_rating": 1, "_id": 0}}
]

df_decade = to_df(col.aggregate(pipeline))
display(df_decade)

,n,avg_rating,decade
0,241,7.248548,1930
1,332,7.247892,1940
2,617,7.164668,1950
3,807,7.104585,1960
4,1166,7.030189,1970
5,1956,6.661912,1980
6,3558,6.619505,1990
7,7195,6.564322,2000
8,5315,6.542220,2010


### 7) Top directores por nº de películas
**Enunciado:** Devuelve 10 directores con más películas según `directors`.

**Pista:** `$unwind: '$directors'` + `$group`.


In [9]:
pipeline = [
    {"$unwind": "$directors"}, # Descomponer array
    {"$group": {"_id": "$directors", "n": {"$sum": 1}}},
    {"$sort": {"n": -1}},
    {"$limit": 10},
    {"$project": {"director": "$_id", "n": 1, "_id": 0}}
]

df_directors = to_df(col.aggregate(pipeline))
display(df_directors)

,n,director
0,40,Woody Allen
1,32,Martin Scorsese
2,31,Takashi Miike
3,29,Steven Spielberg
4,29,John Ford
5,29,Sidney Lumet
6,27,Michael Apted
7,27,Robert Altman
8,27,Spike Lee
9,27,Clint Eastwood


### 8) Top actores por apariciones
**Enunciado:** Devuelve 10 actores con más apariciones según `cast`.

**Pista:** `$unwind: '$cast'`.


In [10]:
pipeline = [
    {"$unwind": "$cast"}, # Descomponer array de actores
    {"$group": {"_id": "$cast", "n": {"$sum": 1}}},
    {"$sort": {"n": -1}},
    {"$limit": 10},
    {"$project": {"actor": "$_id", "n": 1, "_id": 0}}
]

df_actors = to_df(col.aggregate(pipeline))
display(df_actors)


,n,actor
0,67,Gèrard Depardieu
1,58,Robert De Niro
2,51,Michael Caine
3,49,Bruce Willis
4,48,Samuel L. Jackson
5,48,Morgan Freeman
6,47,Christopher Plummer
7,46,Gene Hackman
8,45,Nicolas Cage
9,45,Max von Sydow


### 9) $lookup con `comments`: películas con más comentarios
**Enunciado:** Usa `$lookup` con la colección `comments` para contar comentarios por película y devolver top 10.

**Pista:** `$lookup` + `$addFields: { n_comments: { $size: '$comments' }}`.


In [11]:
# Aseguramos índice para velocidad (opcional pero recomendado)
db.comments.create_index("movie_id")

pipeline = [
    {"$lookup": {
        "from": "comments",
        "localField": "_id",
        "foreignField": "movie_id",
        "as": "comentarios"
    }},
    {"$project": {
        "year": 1,
        "title": 1,
        "n_comments": {"$size": "$comentarios"} # Contamos el tamaño del array generado
    }},
    {"$sort": {"n_comments": -1}},
    {"$limit": 10}
]

df_comments = to_df(col.aggregate(pipeline))
display(df_comments)

,_id,year,title,n_comments
0,573a13bff29313caabd5e91e,2009,The Taking of Pelham 1 2 3,161
1,573a13abf29313caabd25582,2004,50 First Dates,158
2,573a13a3f29313caabd0d1e3,2001,Ocean's Eleven,158
3,573a13a5f29313caabd159a9,2002,About a Boy,158
4,573a13b3f29313caabd3b647,2009,Terminator Salvation,158
5,573a13bcf29313caabd57db6,2009,Sherlock Holmes,157
6,573a139bf29313caabcf3a45,1999,The Mummy,157
7,573a13b0f29313caabd3505e,2008,Hellboy II: The Golden Army,155
8,573a13acf29313caabd289b3,2004,Anchorman: The Legend of Ron Burgundy,154
9,573a13a0f29313caabd05ae1,2001,The Mummy Returns,154


### 10) Búsqueda por texto en título con regex + ranking por rating
**Enunciado:** Busca películas cuyo `title` contenga `love` (case-insensitive) y devuelve top 10 por `imdb.rating`.

**Pista:** `{'title': {'$regex': 'love', '$options': 'i'}}`.

💡 Para búsqueda full-text (`$text`) necesitarías un índice de texto. En Atlas Shell/Compass:
```js
db.movies.createIndex({ title: 'text', plot: 'text' })
```


In [12]:
query = {
    "title": {"$regex": "love", "$options": "i"} # 'i' para ignorar mayúsculas/minúsculas
}

projection = {"title": 1, "year": 1, "imdb": 1, "_id": 0}

# Ordenar por rating desc. Los vacíos saldrán primero.
cursor = col.find(query, projection).sort("imdb.rating", -1).limit(10)

display(to_df(cursor))

,title,year,imdb
0,A Sunday Kind of Love,2015,"{'rating': '', 'votes': '', 'id': 3611112}"
1,Love is God,2003,"{'rating': 8.9, 'votes': 4584, 'id': 367495}"
2,Dr. Strangelove or: How I Learned to Stop Worr...,1964,"{'rating': 8.5, 'votes': 312300, 'id': 57012}"
3,Eloquent Nude: The Love and Legacy of Edward W...,2007,"{'rating': 8.3, 'votes': 38, 'id': 1204893}"
4,Love and Doves,1985,"{'rating': 8.3, 'votes': 1625, 'id': 87650}"
5,Comrades: Almost a Love Story,1996,"{'rating': 8.2, 'votes': 2871, 'id': 117905}"
6,The Crucified Lovers,1954,"{'rating': 8.2, 'votes': 2002, 'id': 46851}"
7,Sexaholix... A Love Story,2002,"{'rating': 8.2, 'votes': 372, 'id': 318062}"
8,Love,1971,"{'rating': 8.1, 'votes': 1055, 'id': 67814}"
9,In the Mood for Love,2000,"{'rating': 8.1, 'votes': 67292, 'id': 118694}"
